# Digitized-Counterdiabatic QAOA (DC-QAOA) with CUDA-Q

**DC-QAOA** (Chandarana *et al.*, *Digitized-counterdiabatic quantum approximate optimization algorithm*) adds a **counterdiabatic** term to each QAOA layer. These extra single-qubit rotations suppress unwanted transitions, so DC-QAOA usually converges with **fewer layers** than standard QAOA.

Here we use it for **molecular docking**, following Ding *et al.*, *Molecular docking via quantum approximate optimization algorithm*. Docking a ligand into a protein is mapped to finding the **maximum vertex-weighted clique** in a *binding interaction graph* (BIG):

- each **node** is a possible ligand–protein contact pair, weighted by how favourable it is;
- an **edge** connects two contacts that are geometrically compatible;
- the best pose is the set of mutually compatible contacts (a clique) with the largest total weight.

**Kernel:** run `00_jupyter_kernel_from_container.ipynb` once to create the CUDA-Q kernels, then select **Python (CUDA-Q GPU)** in the top-right corner. To run on CPU instead, select **Python (CUDA-Q CPU)** and change the target in the first code cell to `cudaq.set_target("qpp-cpu")`.

## 1. Imports and target

In [ ]:
import cudaq
from cudaq import spin
import numpy as np

cudaq.set_target("nvidia")
print("Target:", cudaq.get_target().name)

## 2. The binding interaction graph
**BIG 1** has 6 nodes, 12 edges, node weights $w_i$, a constraint penalty $P = 6$ and 3 layers. Every pair of nodes that is **not** connected (a *non-edge*) cannot be in the same clique.

**BIG 2** (8 nodes, 8 layers) is included as a comment. It is a more expensive simulation.

In [ ]:
# BIG 1
nodes = [0, 1, 2, 3, 4, 5]
qubit_num = len(nodes)
edges = [[0, 1], [0, 2], [0, 4], [0, 5], [1, 2], [1, 3], [1, 5], [2, 3], [2, 4],
         [3, 4], [3, 5], [4, 5]]
non_edges = [
    [u, v] for u in nodes for v in nodes if u < v and [u, v] not in edges
]

print('Edges: ', edges)
print('Non-Edges: ', non_edges)

weights = [0.6686, 0.6686, 0.6686, 0.1453, 0.1453, 0.1453]
penalty = 6.0
num_layers = 3

# BIG 2 (more expensive simulation)
# nodes = [0, 1, 2, 3, 4, 5, 6, 7]
# qubit_num = len(nodes)
# edges = [[0,1],[0,2],[0,5],[0,6],[0,7],[1,2],[1,4],[1,6],[1,7],[2,4],[2,5],[2,7],[3,4],[3,5],[3,6],
#          [4,5],[4,6],[5,6]]
# non_edges = [[u,v] for u in nodes for v in nodes if u<v and [u,v] not in edges]
# weights = [0.6686,0.6686,0.6886,0.1091,0.0770,0.0770,0.0770,0.0770]
# penalty = 8.0
# num_layers = 8

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.Graph()
G.add_nodes_from(nodes)
G.add_edges_from(edges)
pos = nx.circular_layout(G)
nx.draw(G, pos, with_labels=True, node_color="lightblue", node_size=700)
nx.draw_networkx_labels(G, {n: (x * 1.25, y * 1.25) for n, (x, y) in pos.items()},
                        labels={n: f"w={w}" for n, w in zip(nodes, weights)}, font_size=8)
plt.show()

## 3. Cost Hamiltonian
With $x_i = (1 - Z_i)/2 \in \{0,1\}$ meaning "node $i$ is in the clique":

$$H = \frac{1}{2}\sum_i w_i\,(Z_i - I) \;+\; \frac{P}{4}\sum_{(u,v)\,\notin\,E}\big(Z_uZ_v - Z_u - Z_v + I\big)$$

- The first term rewards selecting heavy nodes: it contributes $-w_i$ when $x_i=1$.
- The second term adds a penalty $P$ whenever **both** ends of a non-edge are selected, which would break the clique.

As in the other notebooks, the Hamiltonian is split into coefficients and full-length Pauli words for `exp_pauli`.

In [ ]:
def ham_clique(penalty, nodes, weights, non_edges) -> cudaq.SpinOperator:
    spin_ham = 0
    for wt, node in zip(weights, nodes):
        spin_ham += 0.5 * wt * spin.z(node)
        spin_ham -= 0.5 * wt * spin.i(node)

    for non_edge in non_edges:
        u, v = (non_edge[0], non_edge[1])
        spin_ham += penalty / 4.0 * (spin.z(u) * spin.z(v) - spin.z(u) -
                                     spin.z(v) + spin.i(u) * spin.i(v))
    return spin_ham


# Collect coefficients from a spin operator so we can pass them to a kernel
def term_coefficients(ham: cudaq.SpinOperator) -> list[complex]:
    result = []
    for term in ham:
        result.append(term.evaluate_coefficient())
    return result


# Collect Pauli words from a spin operator so we can pass them to a kernel
def term_words(ham: cudaq.SpinOperator) -> list[str]:
    # The kernel uses these words to apply exp_pauli to the entire state,
    # so each Pauli word must cover the entire space.
    result = []
    for term in ham:
        result.append(term.get_pauli_word(qubit_num))
    return result


ham = ham_clique(penalty, nodes, weights, non_edges)
print(ham)

coef = term_coefficients(ham)
words = term_words(ham)

print(coef)
print(words)

## 4. The DC-QAOA kernel
Starting from $|+\rangle^{\otimes n}$, each of the `num_layers` layers applies:

1. the **cost unitary**: one `exp_pauli` per Hamiltonian term, each with its **own** angle;
2. the **mixer**: $R_x(\beta_j)$ on every qubit;
3. the **counterdiabatic term**: $R_y(\alpha_j)$ on every qubit.

Remove step 3 to get conventional QAOA (with per-term angles).

In [ ]:
@cudaq.kernel
def dc_qaoa(qubit_num: int, num_layers: int, thetas: list[float],
            coef: list[complex], words: list[cudaq.pauli_word]):

    qubits = cudaq.qvector(qubit_num)
    h(qubits)

    count = 0
    for p in range(num_layers):

        for i in range(len(coef)):
            exp_pauli(thetas[count] * coef[i].real, qubits, words[i])
            count += 1

        for j in range(qubit_num):
            rx(thetas[count], qubits[j])
            count += 1

        # Comment out this loop for conventional QAOA
        for k in range(qubit_num):
            ry(thetas[count], qubits[k])
            count += 1

## 5. Optimizer and parameters
Each layer has `len(coef)` cost angles, `qubit_num` mixer angles and `qubit_num` counterdiabatic angles:

$$N_\text{params} = (2\,n_\text{qubits} + N_\text{terms}) \times N_\text{layers}$$

The parameters are initialised uniformly in $[-\pi/8, \pi/8]$ and optimised with **Nelder–Mead**. Seeds are fixed for reproducibility.

In [ ]:
optimizer = cudaq.optimizers.NelderMead()

np.random.seed(13)
cudaq.set_random_seed(13)

# DC-QAOA
parameter_count = (2 * qubit_num + len(coef)) * num_layers

# Conventional QAOA (if the counterdiabatic loop is removed)
# parameter_count = (qubit_num + len(coef)) * num_layers

print('Total number of parameters: ', parameter_count)
optimizer.initial_parameters = np.random.uniform(-np.pi / 8, np.pi / 8, parameter_count)
print("Initial parameters = ", optimizer.initial_parameters)

## 6. Optimise
The objective is $\langle\psi(\theta)|H|\psi(\theta)\rangle$. Every evaluation is stored in `cost_values` so we can plot the convergence.

In [ ]:
cost_values = []


def objective(parameters):
    cost = cudaq.observe(dc_qaoa, ham, qubit_num, num_layers, parameters, coef,
                         words).expectation()
    cost_values.append(cost)
    return cost


optimal_expectation, optimal_parameters = optimizer.optimize(
    dimensions=parameter_count, function=objective)

print('optimal_expectation =', optimal_expectation)
print('optimal_parameters =', optimal_parameters)

## 7. Sample the optimal circuit
The most probable bitstring is the predicted **maximum vertex-weighted clique**: qubits measured as `1` are the contacts selected for the docking pose.

In [ ]:
shots = 200000

counts = cudaq.sample(dc_qaoa, qubit_num, num_layers, optimal_parameters, coef, words,
                      shots_count=shots)
print(counts)

print('The MVWCP is given by the partition: ', counts.most_probable())

## 8. Convergence

In [ ]:
x_values = list(range(len(cost_values)))
y_values = cost_values

plt.plot(x_values, y_values)
plt.xlabel("Epochs")
plt.ylabel("Cost Value")
plt.title("DC-QAOA convergence")
plt.show()

## 9. Plot the selected clique

In [ ]:
best = counts.most_probable()
clique = [n for n in nodes if best[n] == "1"]
clique_edges = [(u, v) for u, v in edges if u in clique and v in clique]
print("Selected nodes:", clique, " total weight:", round(sum(weights[n] for n in clique), 4))

plt.figure(figsize=(6, 6))
nx.draw_networkx_nodes(G, pos, node_color=["red" if n in clique else "lightgray" for n in nodes],
                       edgecolors="black", node_size=1000)
nx.draw_networkx_edges(G, pos, width=1.5, edge_color="gray", alpha=0.5)
nx.draw_networkx_edges(G, pos, edgelist=clique_edges, width=3, edge_color="red")
nx.draw_networkx_labels(G, pos, font_weight="bold")
plt.title("Maximum vertex-weighted clique (red)")
plt.axis("off")
plt.show()